# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Softwareguy786/ML_INTERNSHIP_01/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*    The queue ranks every page by an opportunity score: the model's predicted probability of decline multiplied by log-scaled visibility (90-day impressions). Ranking on decline probability alone sent editors to dead pages with 1–11 impressions — useless work. Multiplying by visibility sends them to pages that are declining and have an audience worth recovering.

    Every row carries a reason code so a human sees why it was flagged: STALE_180D, STALE_90D, OLD_CONTENT, LOW_CTR, HIGH_VISIBILITY, LOW_TRAFFIC. Rows also get one action tier: REFRESH_NOW (prob ≥ 0.65 and impressions ≥ 100), REVIEW_OR_PRUNE (prob ≥ 0.65, low traffic), PROTECT (prob ≤ 0.35, high traffic), MONITOR (everything else). The split: 2,406 REFRESH_NOW, 448 REVIEW_OR_PRUNE, 290 PROTECT, 26,856 MONITOR. This is decision-support — a human reads the top N, not all 30,000.

*

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('/content/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

features = ['content_age_days','days_since_last_update','impressions_90d',
            'clicks_90d','word_count','ctr']
X = df[features].fillna(0)
y = df['is_declining']

model = LogisticRegression(max_iter=1000).fit(X, y)
df['refresh_prob'] = model.predict_proba(X)[:, 1]

# Opportunity = probability of decline × visibility (log-scaled)
df['opportunity'] = df['refresh_prob'] * np.log1p(df['impressions_90d'])

def reason_codes(row):
    tags = []
    if row['days_since_last_update'] > 180: tags.append('STALE_180D')
    if row['days_since_last_update'] > 90:  tags.append('STALE_90D')
    if row['content_age_days'] > 365:       tags.append('OLD_CONTENT')
    if row['ctr'] < df['ctr'].median():     tags.append('LOW_CTR')
    if row['impressions_90d'] > df['impressions_90d'].median(): tags.append('HIGH_VISIBILITY')
    if row['impressions_90d'] < 50:         tags.append('LOW_TRAFFIC')
    return ','.join(tags) if tags else 'NO_TAG'

df['reason_codes'] = df.apply(reason_codes, axis=1)

# Action tier
def action(row):
    if row['refresh_prob'] >= 0.65 and row['impressions_90d'] >= 100:
        return 'REFRESH_NOW'
    if row['refresh_prob'] >= 0.65:
        return 'REVIEW_OR_PRUNE'
    if row['refresh_prob'] <= 0.35 and row['impressions_90d'] >= 500:
        return 'PROTECT'
    return 'MONITOR'

df['action'] = df.apply(action, axis=1)

queue = (df[['content_id','opportunity','refresh_prob','action','reason_codes',
             'days_since_last_update','impressions_90d','ctr']]
         .sort_values('opportunity', ascending=False).reset_index(drop=True))
queue['rank'] = queue.index + 1
queue = queue[['rank','content_id','opportunity','refresh_prob','action','reason_codes',
               'days_since_last_update','impressions_90d','ctr']]

print("Queue size:", len(queue))
print("\nAction counts:")
print(df['action'].value_counts())
print("\nTop 10:")
print(queue.head(10).round(3).to_string(index=False))

Queue size: 30000

Action counts:
action
MONITOR            26856
REFRESH_NOW         2406
REVIEW_OR_PRUNE      448
PROTECT              290
Name: count, dtype: int64

Top 10:
 rank           content_id  opportunity  refresh_prob      action                      reason_codes  days_since_last_update  impressions_90d  ctr
    1 content_b28d1efd668f        9.400         0.748 REFRESH_NOW STALE_90D,LOW_CTR,HIGH_VISIBILITY                     104           286608 0.06
    2 content_813e88069237        8.955         0.724 REFRESH_NOW STALE_90D,LOW_CTR,HIGH_VISIBILITY                     104           233561 0.06
    3 content_36ff89c8214e        8.788         0.698 REFRESH_NOW STALE_90D,LOW_CTR,HIGH_VISIBILITY                     104           295097 0.05
    4 content_a023517539fe        8.694         0.708 REFRESH_NOW           LOW_CTR,HIGH_VISIBILITY                      20           214047 0.01
    5 content_4a6607efcb46        8.663         0.737 REFRESH_NOW STALE_90D,LOW_CTR,HIGH_VISIB

## 2. Intended use and limits

*Who uses it: a content editor or SEO lead with a backlog of pages, who needs to know which to review first. They open the queue, read the top 50–100 REFRESH_NOW rows, and decide page-by-page.

What it answers: "given this snapshot, which pages are most likely to be declining and most worth a human's attention first?"

Where it stops being valid:

    It does not know why a page is declining — algorithm change, lost links, seasonality, or content rot all look the same to the model.

    Grouped-split F1 in Week 6 ranged 0.457–0.799 across client groups — performance is unstable by client, so the queue is directional, not precise.

    The label is a 90-day trend proxy, not an observed refresh outcome. No page has been refreshed and measured.

    The queue does not measure the effect of refreshing. It only orders the backlog.*

In [4]:
# Snapshot of how much of the queue a human would actually work through
top_n = [10, 25, 50, 100, 250]
for n in top_n:
    sub = queue.head(n)
    print(f"Top {n:>3}: REFRESH_NOW={(sub['action']=='REFRESH_NOW').sum()}, "
          f"median impressions={int(sub['impressions_90d'].median())}, "
          f"median CTR={round(sub['ctr'].median(),3)}")

Top  10: REFRESH_NOW=9, median impressions=225918, median CTR=0.04
Top  25: REFRESH_NOW=24, median impressions=130617, median CTR=0.04
Top  50: REFRESH_NOW=46, median impressions=109572, median CTR=0.055
Top 100: REFRESH_NOW=85, median impressions=79706, median CTR=0.065
Top 250: REFRESH_NOW=187, median impressions=48607, median CTR=0.09


## 3. Human review + the no-go list

*Counts from the flagging rule: 448 REVIEW_OR_PRUNE rows — every one goes to a human before pruning. 1,283 REFRESH_NOW rows sit in the uncertain band (prob 0.45–0.65) or have high CTR despite being flagged — those get a human look before work starts. The remaining ~1,100 REFRESH_NOW rows are clean enough to queue directly. The 20,680 MONITOR rows are not urgent and don't need review unless an editor is looking for a backlog.*

In [5]:
# Flag the rows that MUST go to a human — the model is uncertain or the case is ambiguous
queue['needs_human'] = (
    queue['refresh_prob'].between(0.45, 0.65)          # uncertain zone
    | (queue['reason_codes'] == 'NO_TAG')               # no explanation
    | (queue['action'] == 'REVIEW_OR_PRUNE')            # pruning = always human
    | ((queue['action'] == 'REFRESH_NOW') & (queue['ctr'] >= 0.10))  # high CTR but flagged
)

print("Rows needing human review before action:", queue['needs_human'].sum())
print("\nBy action:")
print(queue.groupby('action')['needs_human'].agg(['sum','count']))
print("\nSample of REVIEW_OR_PRUNE rows (pruning is always human-decided):")
print(queue[queue['action']=='REVIEW_OR_PRUNE']
      .head(5)[['rank','content_id','refresh_prob','impressions_90d','ctr','reason_codes']]
      .round(3).to_string(index=False))

Rows needing human review before action: 22411

By action:
                   sum  count
action                       
MONITOR          20680  26856
PROTECT              0    290
REFRESH_NOW       1283   2406
REVIEW_OR_PRUNE    448    448

Sample of REVIEW_OR_PRUNE rows (pruning is always human-decided):
 rank           content_id  refresh_prob  impressions_90d  ctr                 reason_codes
14856 content_7a888d3d99c8         0.753               95  0.0 STALE_180D,STALE_90D,LOW_CTR
15135 content_07ce98c6085a         0.762               85  0.0 STALE_180D,STALE_90D,LOW_CTR
15449 content_bbca724138f2         0.772               75  0.0 STALE_180D,STALE_90D,LOW_CTR
16306 content_b8366b9ef4e8         0.769               64  0.0 STALE_180D,STALE_90D,LOW_CTR
16319 content_30eb41dff556         0.722               84  0.0 STALE_180D,STALE_90D,LOW_CTR


## 4. Monitoring / retrain triggers

*What tells me the queue went stale:

    The share of REFRESH_NOW rows drops below ~5% or rises above ~15% of the queue — either the data shifted or the threshold is wrong.

    The grouped-split F1 drops below 0.55 on a fresh data pull — the model is no longer better than a coin flip on some client groups.

    The median impressions of the top 100 falls below ~20,000 — the queue is drifting back toward dead pages.

    The reason-code mix changes sharply (e.g. STALE_180D drops from ~50% to <20% of the queue) — the data contract may have changed.

Retrain when any two of those fire, or every 90 days, whichever comes first. Retraining means rerunning w05_model.ipynb on a fresh pull, re-running w06_validation_audit.ipynb to confirm the grouped F1 hasn't moved, then regenerating this queue.*

In [8]:
baseline = {
    'refresh_now_share':          round((queue['action']=='REFRESH_NOW').mean(), 3),
    'median_impressions_top100':  int(queue.head(100)['impressions_90d'].median()),
    'stale_90d_share_top100':     round(queue.head(100)['reason_codes'].str.contains('STALE_90D').mean(), 3),
    'low_ctr_share_top100':       round(queue.head(100)['reason_codes'].str.contains('LOW_CTR').mean(), 3),
    'grouped_f1_w06':             0.661,
}

print("Baseline monitoring numbers (compare future pulls against these):")
for k, v in baseline.items():
    print(f"  {k}: {v}")

print("\nAlert thresholds (set around the observed baseline):")
print("  refresh_now_share outside 0.04–0.15      → investigate")
print("  median_impressions_top100 < 20,000       → investigate")
print("  stale_90d_share_top100 outside 0.60–0.98 → investigate")
print("  low_ctr_share_top100 outside 0.25–0.75   → investigate")
print("  grouped_f1 (fresh w06 re-run) < 0.55     → retrain")

Baseline monitoring numbers (compare future pulls against these):
  refresh_now_share: 0.08
  median_impressions_top100: 79706
  stale_90d_share_top100: 0.88
  low_ctr_share_top100: 0.5
  grouped_f1_w06: 0.661

Alert thresholds (set around the observed baseline):
  refresh_now_share outside 0.04–0.15      → investigate
  median_impressions_top100 < 20,000       → investigate
  stale_90d_share_top100 outside 0.60–0.98 → investigate
  low_ctr_share_top100 outside 0.25–0.75   → investigate
  grouped_f1 (fresh w06 re-run) < 0.55     → retrain


## 5. Exports for the paper

*Exports go to work/outputs/ in the repo. Four files: the full ranked queue (CSV), the top 100 for the paper's action playbook table, a summary of action counts and reason-code shares (for charts), and a one-row baseline metrics file (for the Results section). These are the only artifacts the paper needs — everything else stays inside the notebook.*

In [9]:
import os, json

out_dir = '/content/outputs'
os.makedirs(out_dir, exist_ok=True)

# 1. Full queue
queue.to_csv(f'{out_dir}/refresh_queue_full.csv', index=False)

# 2. Top 100 for the paper table
queue.head(100).to_csv(f'{out_dir}/refresh_queue_top100.csv', index=False)

# 3. Summary for charts
summary = pd.DataFrame({
    'action': queue['action'].value_counts().index,
    'count':  queue['action'].value_counts().values,
})
summary['share'] = (summary['count'] / len(queue)).round(3)
summary.to_csv(f'{out_dir}/action_summary.csv', index=False)

reason_summary = pd.DataFrame({
    'reason_code': ['STALE_90D','STALE_180D','LOW_CTR','HIGH_VISIBILITY','OLD_CONTENT','LOW_TRAFFIC'],
    'share_top100': [
        round(queue.head(100)['reason_codes'].str.contains(c).mean(), 3)
        for c in ['STALE_90D','STALE_180D','LOW_CTR','HIGH_VISIBILITY','OLD_CONTENT','LOW_TRAFFIC']
    ],
})
reason_summary.to_csv(f'{out_dir}/reason_code_summary.csv', index=False)

# 4. Baseline metrics for Results section
baseline_metrics = {
    'random_split_f1_w05':  0.680,
    'grouped_split_f1_w06': 0.661,
    'grouped_f1_min_fold':  0.457,
    'grouped_f1_max_fold':  0.799,
    'baseline_rule_f1':     0.009,
    'queue_size':           len(queue),
    'refresh_now_count':    int((queue['action']=='REFRESH_NOW').sum()),
    'refresh_now_share':    round((queue['action']=='REFRESH_NOW').mean(), 3),
    'median_impressions_top100': int(queue.head(100)['impressions_90d'].median()),
}
with open(f'{out_dir}/baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2)

print("Wrote files:")
for fn in sorted(os.listdir(out_dir)):
    print(" ", fn, "-", os.path.getsize(f'{out_dir}/{fn}'), "bytes")

Wrote files:
  action_summary.csv - 104 bytes
  baseline_metrics.json - 278 bytes
  reason_code_summary.csv - 120 bytes
  refresh_queue_full.csv - 3348542 bytes
  refresh_queue_top100.csv - 12505 bytes


In [10]:
import shutil
from google.colab import files

shutil.make_archive('/content/outputs', 'zip', '/content/outputs')
files.download('/content/outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.